# Breed Dog Image Classification using AWS Sagemaker

### Project Description
This project focuses on solving a complex computer vision problem: identifying dog breeds from images. With over 130 breeds in the dataset, the challenge lies in capturing fine-grained visual features that distinguish highly similar breeds (low inter-class variation) while handling various poses and backgrounds (high intra-class variation). The solution is built as an end-to-end Machine Learning pipeline using Amazon SageMaker.

### Project Objective
The primary goal is to develop a highly accurate image classification model using Transfer Learning. By leveraging a pre-trained ResNet50 architecture, the project aims to minimize computational costs and training time while achieving professional-grade performance in breed identification. The final model is deployed to a real-time SageMaker Endpoint for scalable inference.

### Tools and Technologies
1. Cloud Platform: Amazon Web Services (AWS).

2. Machine Learning Service: Amazon SageMaker (Training Jobs, Hyperparameter Tuning, Endpoints).

3. Deep Learning Framework: PyTorch (v1.8+).

4. Model Architecture: ResNet50 (Pre-trained on ImageNet).

5. Monitoring & Logging: SageMaker Debugger and Profiler.

6. Programming Language: Python 3.

### Training Strategy
The training workflow follows a rigorous MLOps lifecycle divided into three main stages:

- Data Preparation: Images are organized into S3 channels and preprocessed using PyTorch transforms, including Data Augmentation (rotation, flipping, and color jittering) to improve model generalization.

- Hyperparameter Optimization (HPO): A dedicated tuning job (hpo.py) executes multiple trials to find the optimal combination of Learning Rate and Batch Size by monitoring the Validation Loss.

- Fine-Tuning & Monitoring: The best hyperparameters are used in a final training job (train_model.py). This stage integrates SageMaker Debugger to monitor gradients and Profiler to analyze hardware utilization, ensuring the model trains efficiently without overfitting.

In [ ]:
# Install any packages that you might need
# For instance, you will need the smdebug package
!pip install smdebug
!pip install opencv-python

In [ ]:
# Import any packages that you might need
# For instance you will need Boto3 and Sagemaker
import sagemaker
import boto3
import numpy as np
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sagemaker.pytorch import PyTorch
import sagemaker

## Dataset
### Overview
The dataset used for this project is a comprehensive collection of dog images categorized by breed. It is a popular benchmark in the computer vision community for fine-grained image classification. Unlike general classification tasks (e.g., distinguishing between a car and a dog), this dataset requires the model to identify subtle morphological differences between closely related canine subspecies.

### Dataset Statistics
- Total Number of Classes: 133 distinct dog breeds (e.g., Affenpinscher, Airedale Terrier, Chihuahua, etc.).

- Data Split: The images are physically partitioned into three subsets to ensure a robust evaluation:

- Training Set: Used for the model to learn visual features.

- Validation Set: Used during Hyperparameter Tuning (HPO) to select the best model configuration and prevent overfitting.

- Test Set: Reserved for final evaluation to measure the model's true generalization capability on unseen data.

- Total Size: Approximately 1.15 GB of image data.

In [ ]:
#TODO: Fetch and upload the data to AWS S3
# Command to download and unzip data
!wget https://s3-us-west-1.amazonaws.com/udacity-aind/dog-project/dogImages.zip
!unzip dogImages.zip
current_path = os.getcwd()
train_path = os.path.join(current_path, 'train')
val_path = os.path.join(current_path, 'valid')
test_path = os.path.join(current_path, 'test')
BASE_FOLDER=current_path
train_labels = os.listdir(train_path)
valid_labels = os.listdir(val_path)
test_labels = os.listdir(test_path)
print(len(train_labels))
print(len(valid_labels))
print(len(test_labels))

In [ ]:
from collections import defaultdict

limit = 20
def count_images_per_class(directory):
    counts = {}
    for label in os.listdir(directory):
        label_path = os.path.join(directory, label)
        if os.path.isdir(label_path):
            counts[label] = len(os.listdir(label_path))
    return counts

train_counts = count_images_per_class(train_path)
val_counts = count_images_per_class(val_path)
test_counts = count_images_per_class(test_path)
for label in list(train_counts.keys())[:limit]:
    print(f"Clase: {label}, Train: {train_counts[label]}, Val: {val_counts.get(label, 0)}, Test: {test_counts.get(label, 0)}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os
import random

dataSets = os.listdir(BASE_FOLDER)
num_datasets = len(dataSets)
fig, axes = plt.subplots(num_datasets, 5, figsize=(20, 4*num_datasets))
fig.tight_layout()
plt.subplots_adjust(top=0.9)
for i, dataset in enumerate(dataSets):
    folders = os.listdir(os.path.join(BASE_FOLDER, dataset))
    for j in range(5):
        folder = random.choice(folders)
        files = os.listdir(os.path.join(BASE_FOLDER, dataset, folder))
        img_path = os.path.join(BASE_FOLDER, dataset, folder, random.choice(files))
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if num_datasets > 1:
            axes[i, j].imshow(img)
            axes[i, j].set_title(f'{dataset}-{folder}')
            axes[i, j].axis('off')
        else:
            axes[j].imshow(img)
            axes[j].set_title(f'{dataset}-{folder}')
            axes[j].axis('off')

plt.show()

In [ ]:
import cv2
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

def inspect_dataset(data_dir: list):
  #print(data_dir)
  data={}
  for dir in data_dir:
    #print(dir)
    class_names = os.listdir(dir)
    num_classes = len(class_names)
    print(f"Folder {dir} Total Classes: {num_classes}")
    # show examples of classes
    print(f"Examples of classes: {class_names[:10]}")

    # count images per class
    class_counts = {}
    total_images = 0
    for class_name in class_names:
      class_dir = os.path.join(dir, class_name)
      if os.path.isdir(dir):
          img_count = len(os.listdir(class_dir))
          class_counts[class_name] = img_count
          total_images += img_count
    print(f"Total images: {total_images}")
    data[dir]=class_counts

  data_list = []
  for category, clases in data.items():
    path_category = Path(category)
    last_folder = path_category.name
    for class_label, value in clases.items():
        data_list.append({
            'Category': last_folder,
            'Class': class_label,
            'Value': value
        })
  # Create a dataframe
  df = pd.DataFrame(data_list)

  # Sum the values per class and get the top 10 classes
  valores_por_clase = df.groupby('Class')['Value'].sum().reset_index()
  top10_clases = valores_por_clase.sort_values('Value', ascending=False).head(20)['Class'].tolist()

  # Filtrar el DataFrame para incluir solo las top 10 clases
  df_top10 = df[df['Class'].isin(top10_clases)]

  # Pivot the DataFrame to have categories as columns and classes as index
  df_ancho = df_top10.pivot(index='Class', columns='Category', values='Value')

  # Create figure
  plt.figure(figsize=(12, 6))
  sns.set_style("whitegrid")

  # stacked bar plot
  df_ancho.plot(kind='bar', stacked=True, colormap='viridis')

  # Customize the plot
  plt.title('Top 10 classes with stacked categories', fontsize=16)
  plt.xlabel('Classes', fontsize=12)
  plt.ylabel('Values', fontsize=12)
  plt.xticks(rotation=45, ha='right')
  plt.legend(title='Category')

  # Ajustar el diseño y mostrar la gráfica
  plt.tight_layout()
  plt.show()
  return class_names, num_classes

inspect_dataset([test_path,train_path,val_path])


print('-----------------------------------------------------------------\n\n')
print('Diferencia entre categorias test-train: ')
print(len(set(test_labels).difference(train_labels)))
print('Diferencia entre categorias val-train: ')
print(len(set(valid_labels).difference(train_labels)))

In [ ]:
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()

In [ ]:
bucket = sagemaker_session.default_bucket()
# 3. The prefix (the folder in S3 where the data will be stored)
prefix_base = 'dog-breed-classification'
prefix = f'{prefix_base}/data'

# 4. Local path where you have the dataset unzipped
local_data_path = './dogImages'

print(f"Uploading data to S3: s3://{bucket}/{prefix}...")

s3_data_uri = sagemaker_session.upload_data(
    path=local_data_path, 
    bucket=bucket, 
    key_prefix=prefix
)

print(f"Upload complete. S3 URI for Training Jobs: {s3_data_uri}")

## Hyperparameter Tuning
**TODO:** This is the part where you will finetune a pretrained model with hyperparameter tuning. Remember that you have to tune a minimum of two hyperparameters. However you are encouraged to tune more. You are also encouraged to explain why you chose to tune those particular hyperparameters and the ranges.

**Note:** You will need to use the `hpo.py` script to perform hyperparameter tuning.

In [ ]:
#TODO: Declare your HP ranges, metrics etc.
import sagemaker
from sagemaker.tuner import (
    IntegerParameter,
    CategoricalParameter,
    ContinuousParameter,
    HyperparameterTuner,
)

hyperparameter_ranges = {
    "lr": ContinuousParameter(0.001, 0.1),
    "batch-size": CategoricalParameter([32, 64, 128, 256, 512]),
}

objective_metric_name = "average test loss"
objective_type = "Minimize"
metric_definitions = [{"Name": "average test loss", "Regex": "Test set: Average loss: ([0-9\\.]+)"}]

In [ ]:
from sagemaker.pytorch import PyTorch

estimator = PyTorch(
    entry_point="hpo.py",
    source_code_dir="code",
    base_job_name="pytorch-hpo-dog-breed-classification",
    role=role,
    py_version='py38',
    framework_version="1.9",
    instance_count=1,
    instance_type="ml.m5.2xlarge"
)

tuner = HyperparameterTuner(
    estimator,
    objective_metric_name,
    hyperparameter_ranges,
    metric_definitions,
    max_jobs=4,
    max_parallel_jobs=2,
    base_tuning_job_name = 'pytorch-dog-breed-hpo-tuning',
    objective_type=objective_type,
)

In [ ]:
# TODO: Fit your HP Tuner
tuner.fit({"data": s3_data_uri}, wait=True) # TODO: Remember to include your data channels, por ejemplo data --> SM_CHANNEL_DATA

In [ ]:
tuner.describe()

In [ ]:
tuner.best_training_job()

In [ ]:
# Get the best estimators and the best HPs
best_estimator = tuner.best_estimator()

#Get the hyperparameters of the best trained model
best_hyperparameters = best_estimator.hyperparameters()

In [ ]:
best_hyperparameters

In [ ]:
tuner.analytics().dataframe()

In [ ]:
best_hyperparameter_params = {
    "lr": best_hyperparameters["lr"],
    "batch_size": best_hyperparameters["batch_size"].replace('"', '')  # Remove quotes if it's a string,
}

In [ ]:
best_hyperparameter_params

## Model Profiling and Debugging
TODO: Using the best hyperparameters, create and finetune a new model

**Note:** You will need to use the `train_model.py` script to perform model profiling and debugging.

In [ ]:
!pip install "smdebug==1.0.34" "protobuf<=3.20.3" "bokeh<3.0.0"

import os
import sys

os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

# Parcheamos Bokeh para evitar el error 'plot_height' en TimelineCharts
import bokeh.plotting
if not hasattr(bokeh.plotting.Figure, 'plot_height'):
    bokeh.plotting.Figure.plot_height = property(lambda self: self.height, 
                                                lambda self, val: setattr(self, 'height', val))

In [ ]:
# TODO: Set up debugging and profiling rules and hooks
from sagemaker.debugger import (
    Rule, 
    rule_configs, 
    ProfilerRule, 
    DebuggerHookConfig, 
    ProfilerConfig, 
    FrameworkProfile
    )


rules = [
    Rule.sagemaker(rule_configs.loss_not_decreasing()),
    ProfilerRule.sagemaker(rule_configs.LowGPUUtilization()),
    ProfilerRule.sagemaker(rule_configs.ProfilerReport()),
    Rule.sagemaker(rule_configs.vanishing_gradient()),
    Rule.sagemaker(rule_configs.overfit()),
    Rule.sagemaker(rule_configs.overtraining()),
    Rule.sagemaker(rule_configs.poor_weight_initialization()),
]

profiler_config = ProfilerConfig(
    system_monitor_interval_millis=500, framework_profile_params=FrameworkProfile(num_steps=10)
)
debugger_config = DebuggerHookConfig(
    hook_parameters={"train.save_interval": "100", "eval.save_interval": "10"}
)
output_location = f's3://{bucket}/{prefix_base}/output/'

In [ ]:
metric_definitions = [
    # loss - training
    {"Name": "train:loss", "Regex": "Phase train - Loss: ([0-9\\.]+)"},
    
    # accuracy - training
    {"Name": "train:accuracy", "Regex": "Phase train - Loss: [0-9\\.]+ Acc: ([0-9\\.]+)"},
    
    # loss - validation
    {"Name": "validation:loss", "Regex": "Phase valid - Loss: ([0-9\\.]+)"},
    
    # accuracy - validation
    {"Name": "validation:accuracy", "Regex": "Phase valid - Loss: [0-9\\.]+ Acc: ([0-9\\.]+)"}
]

In [ ]:
estimator = PyTorch(
    role = role,
    instance_count = 1,
    instance_type = "ml.m5.2xlarge",
    entry_point = "train_model.py",
    source_code_dir = "code",
    base_job_name = "smdebugger-train-dog-breed-class-pytorch",
    framework_version = "1.9",
    py_version = "py38",
    output_location = output_location,
    hyperparameters = best_hyperparameter_params,
    debugger_hook_config = debugger_config,
    profiler_config = profiler_config,
    metric_definitions = metric_definitions,
    rules = rules
)

estimator.fit({"data": s3_data_uri}, wait=True)

In [ ]:
import boto3

session = boto3.session.Session()
region = session.region_name

training_job_name = estimator.latest_training_job.name
print(f"Training jobname: {training_job_name}\n")
print(f"Debugger output path: {estimator.latest_job_debugger_artifacts_path()}\n")
print(f"Region: {region}\n")
print("rule_output_path : {}\n".format(estimator.output_path + estimator.latest_training_job.job_name + "/rule-output"))

In [ ]:
# TODO: Plot a debugging output.
from smdebug.trials import create_trial
from smdebug.core.modes import ModeKeys

trial = create_trial(estimator.latest_job_debugger_artifacts_path())

In [ ]:
print(trial.tensor_names())

In [ ]:
import matplotlib.pyplot as plt

def plot_tensor(trial, tensor_name):
    # Obtenemos los pasos (steps) disponibles para ese tensor
    steps = trial.tensor(tensor_name).steps(mode=ModeKeys.TRAIN)
    steps_eval = trial.tensor(tensor_name).steps(mode=ModeKeys.EVAL)
    
    plt.figure(figsize=(10, 6))
    
    # Valores de entrenamiento
    train_values = [trial.tensor(tensor_name).value(s, mode=ModeKeys.TRAIN) for s in steps]
    plt.plot(steps, train_values, label='Train Loss', color='blue')
    
    # Valores de validación
    eval_values = [trial.tensor(tensor_name).value(s, mode=ModeKeys.EVAL) for s in steps_eval]
    plt.plot(steps_eval, eval_values, label='Evaluation Loss', color='red', linestyle='--')
    
    plt.title(f'Analysis of {tensor_name} (Overfitting Detection)')
    plt.xlabel('Steps')
    plt.ylabel('Value')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_tensor(trial, "CrossEntropyLoss_output_0")

In [ ]:
import torch
def plot_accuracy(trial):
    tensor_name = trial.tensor_names(collection="gradients")[0] # Tomamos el primer gradiente
    
    steps = trial.tensor(tensor_name).steps(mode=ModeKeys.TRAIN)
    grad_norms = [torch.norm(torch.tensor(trial.tensor(tensor_name).value(s, mode=ModeKeys.TRAIN))) for s in steps]
    
    plt.figure(figsize=(10, 5))
    plt.plot(steps, grad_norms, color='green')
    plt.title('Norma de los Gradientes (¿Sigue aprendiendo el modelo?)')
    plt.ylabel('Gradient Norm')
    plt.xlabel('Steps')
    plt.show()

In [ ]:
import time

description = estimator.describe_training_job()
for rule in description['DebugRuleConfigurations']:
    rule_name = rule['RuleConfigurationName']
    status = estimator.latest_training_job.rule_job_summary()[0]['RuleEvaluationStatus']
    print(f"Regla: {rule_name} -> Estado: {status}")

In [ ]:
from smdebug.profiler.analysis.notebook_utils.training_job import TrainingJob

tj = TrainingJob(training_job_name, region)
tj.wait_for_sys_profiling_data_to_be_available()

**TODO**: Is there some anomalous behaviour in your debugging output? If so, what is the error and how will you fix it?  
**TODO**: If not, suppose there was an error. What would that error look like and how would you have fixed it?

In [ ]:
# TODO: Display the profiler output
from smdebug.profiler.analysis.notebook_utils.timeline_charts import TimelineCharts
system_metrics_reader = tj.get_systems_metrics_reader()
system_metrics_reader.refresh_event_file_list()

view_timeline_charts = TimelineCharts(
    system_metrics_reader,
    framework_metrics_reader=None,
    select_dimensions=["CPU", "GPU"],
    select_events=["total"],
)

In [ ]:
rule_output_path = estimator.output_path + estimator.latest_training_job.job_name + "/rule-output"
print(f"You will find the profiler report in {rule_output_path}")

In [ ]:
! aws s3 ls {rule_output_path} --recursive

In [ ]:
! aws s3 cp {rule_output_path} ./ --recursive

In [ ]:
import os

# get the autogenerated folder name of profiler report
profiler_report_name = [
    rule["RuleConfigurationName"]
    for rule in estimator.latest_training_job.rule_job_summary()
    if "Profiler" in rule["RuleConfigurationName"]
][0]

In [ ]:
import IPython

IPython.display.HTML(filename=profiler_report_name + "/profiler-output/profiler-report.html")

## Model Deploying

In [ ]:
# TODO: Deploy your model to an endpoint

predictor=estimator.deploy() # TODO: Add your deployment configuration like instance type and number of instances

In [ ]:
from sagemaker.pytorch import PyTorchModel
import time


timestamp = time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())
inference_endpoint_name = f"dog-breed-classifier-ep-{timestamp}"

# 1. Create model in SageMaker and inference script
pytorch_model = PyTorchModel(
    model_data=estimator.model_data,
    role=role,
    entry_point='inference.py',
    source_code_dir = "code",
    framework_version='1.9',
    py_version='py38'
)

# 2. Deploy endpoint
predictor = pytorch_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name=inference_endpoint_name,
    serializer=sagemaker.serializers.IdentitySerializer(content_type="image/jpeg")
)

print(f"Endpoint desplegado en: {predictor.endpoint_name}")

In [ ]:
import os

data_dir = "dogImages/train"
classes = sorted(os.listdir(data_dir))
dog_names = [cls.split('.')[-1].replace('_', ' ') for cls in classes]

In [ ]:
import requests
import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def softmax(x):
    """softmax calculation."""
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

def test_model_with_url(url, ground_truth):
    """
        Download image from URL, perform inference, and display results.
    """
    print(f"\nGround Truth: {ground_truth}")
    
    try:
        # 1. Download the image
        response_img = requests.get(url, timeout=10)
        if response_img.status_code != 200:
            print(f"Error in download: {url}")
            return
        
        image_bytes = response_img.content
        
        # 2. Inference
        # Execute predictor with IdentitySerializer(content_type="image/jpeg")
        response = predictor.predict(image_bytes, initial_args={"ContentType": "image/jpeg"})
        
        # 3. Processing logits and apply softmax
        logits = np.array(response[0])
        probabilities = softmax(logits)
        
        predicted_index = np.argmax(logits)
        predicted_name = dog_names[predicted_index]
        confidence = probabilities[predicted_index]
        
        # 4. Visualization
        img = Image.open(io.BytesIO(image_bytes))
        plt.figure(figsize=(6, 4))
        plt.imshow(img)
        plt.title(f"Predicted: {predicted_name} ({confidence*100:.2f}%)\nReal: {ground_truth}")
        plt.axis('off')
        plt.show()
        
        # 5. Report
        print(f"Result -> Prediction: {predicted_name} | Probability: {confidence:.4f}")
        
    except Exception as e:
        print(f"Error in the test: {e}")

# --- SAMPLES ---
test_samples = [
    {
        "url": "https://s3.amazonaws.com/cdn-origin-etr.akc.org/wp-content/uploads/2017/11/12231410/Affenpinscher-On-White-01.jpg",
        "label": "Affenpinscher"
    },
    {
        "url": "https://s3.amazonaws.com/cdn-origin-etr.akc.org/wp-content/uploads/2017/11/12224408/Golden-Retriever-On-White-01.jpg",
        "label": "Golden Retriever"
    }
]

for sample in test_samples:
    test_model_with_url(sample['url'], sample['label'])

In [ ]:
# TODO: Remember to shutdown/delete your endpoint once your work is done
predictor.delete_endpoint()